# 00 · Setup: Drive, package, GPU, paths, manifests

**GitHub = code · Colab `/content` = fast ephemeral execution · Google Drive (`MyDrive/GEMS_2`) = persistent data and artifacts.**

Run this notebook at the start of every Colab session (or copy its first cell into any other notebook: every notebook here starts with the same idempotent bootstrap).

1. Mount Drive · 2. clone the repository at a pinned commit and `pip install -e .[colab]` · 3. check PyTorch / GPU · 4. record package version + commit · 5. free disk / VRAM · 6. validate data paths · 7. create the workspace folders · 8. stage the core data from Drive to `/content/gems_work` · 9. reload experiment manifests · 10. environment report.

Nothing here trains a model. No credentials are stored in the notebook: a private repository is cloned with a token read from Colab *Secrets* (`GITHUB_TOKEN`).

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'OWNER/REPOSITORY'          # GitHub owner/name of this project
COMMIT = 'main'                    # pin a commit SHA for reproducible experiments
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')       # editable install is not visible to an already-running kernel
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
print(subprocess.run(['git', '-C', '/content/GEMS', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout)

In [ ]:
# PyTorch / GPU and the auto-configuration the trainers will use
from gems_bench.training import gpu
g = gpu.detect()
print(g)
print('autoconfig (256 px, effective batch 16, 29 channels):', gpu.autoconfig(g, patch_px=256, effective_batch=16, n_channels=29))

In [ ]:
# Drive layout: create missing workspace folders on Drive and locally (never deletes anything)
from pathlib import Path
from gems_bench.paths import WORKSPACE_DIRS
for d in WORKSPACE_DIRS:
    (Path(DRIVE_ROOT) / d).mkdir(parents=True, exist_ok=True)
    (Path(colab.LOCAL_ROOT) / d).mkdir(parents=True, exist_ok=True)
required = ['data/raw/geodawn/gems-geodawn-numerical-features.tif', 'data/raw/faults/existing_faults.tif',
            'data/raw/geodawn/example_submission.tif', 'outputs/benchmark/folds', 'outputs/phase4/experiments']
for r in required:
    print(('OK      ' if (Path(DRIVE_ROOT) / r).exists() else 'MISSING ') + r)
print('See docs/GOOGLE_DRIVE_LAYOUT.md for what to upload once from the local project.')

In [ ]:
# Stage the core data to the fast local disk (unchanged files are skipped on later sessions)
staged = colab.stage_core()
staged.groupby('action').bytes.agg(['count', 'sum'])

In [ ]:
# Existing experiment manifests and the model registry (from Drive)
import json
import pandas as pd
from gems_bench.artifacts import ModelRegistry
rows = []
for m in sorted(Path(DRIVE_ROOT, 'experiments').glob('*/manifest.json')):
    j = json.loads(m.read_text())
    rows.append({'experiment': j['experiment_id'], 'kind': j['kind'], 'commit': j.get('git_commit', '')[:10],
                 **{s: v['status'] for s, v in j['stages'].items()}})
display(pd.DataFrame(rows))
display(ModelRegistry(Path(DRIVE_ROOT) / 'model_registry').table())

In [ ]:
colab.environment_report()